# H-T01-E300 정식 5-Fold 검증

H-T01 설정(Tversky α=0.4/β=0.6, SGD, LR 0.01, batch 2)을 고정하고 fold 0~4를 모두 scratch 300 epoch로 학습합니다.

각 fold는 매 epoch checkpoint, 5 epoch마다 full-volume validation, Top-3 ±4 epoch 최종 재검증을 수행합니다. 기존 H-T01-E300 continuation 결과는 건드리지 않습니다.

In [ ]:
from pathlib import Path
import json, subprocess, time
import pandas as pd
from IPython.display import clear_output, display
ROOT=Path("/home/stu03/workspace/model")
DATA_ROOT=Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
RESULTS_ROOT=ROOT/"runs/nnunet"
RUNNER=ROOT/"scripts/kfold/training/run_h_t01_e300_kfold.py"
BATCH_DIR=ROOT/"runs/batches/H-T01-E300-KF"
BATCH_STATUS=BATCH_DIR/"batch_status.json"
LOG_FILE=BATCH_DIR/"kfold.log"
PID_FILE=BATCH_DIR/"kfold.pid"
PLAN="ICHPlansTuneA04B06SgdKFoldV1"
BASE=RESULTS_ROOT/"Dataset001_ICHBinary"/f"nnUNetTrainerICHBaseline__{PLAN}__3d_fullres"
PROGRESS_FILES={f"fold_{i}":BASE/f"fold_{i}/progress.json" for i in range(5)}
print("runner:",RUNNER.exists(),RUNNER)
print("splits:",DATA_ROOT/"nnUNet_preprocessed/Dataset001_ICHBinary/splits_final.json")

## 1. Split 및 GPU 확인

In [ ]:
split_path=DATA_ROOT/"nnUNet_preprocessed/Dataset001_ICHBinary/splits_final.json"
splits=json.loads(split_path.read_text())
for i,s in enumerate(splits):
    print(f"fold {i}: train={len(s['train'])}, val={len(s['val'])}, overlap={len(set(s['train'])&set(s['val']))}")
subprocess.run(["nvidia-smi"],check=False)

## 2. Dry-run

In [ ]:
command=[str(ROOT/".venv/bin/python"),str(RUNNER),"--data-root",str(DATA_ROOT),"--results-root",str(RESULTS_ROOT),"--gpu","0"]
subprocess.run(command+["--dry-run"],cwd=ROOT,check=True)

## 3. Fold 0~4 백그라운드 실행/자동 재개

GPU에 다른 학습이 없는 것을 확인한 뒤 한 번만 실행하세요. 재실행 시 완료 fold는 건너뛰고 `checkpoint_latest.pth`가 있는 fold는 자동 재개합니다.

In [ ]:
BATCH_DIR.mkdir(parents=True,exist_ok=True)
def pid_is_running(pid_file):
    if not pid_file.is_file():
        return False
    try:
        pid = int(pid_file.read_text().strip())
        result = subprocess.run(
            ["ps", "-o", "stat=", "-p", str(pid)],
            capture_output=True, text=True,
        )
        state = result.stdout.strip()
        return result.returncode == 0 and bool(state) and not state.startswith("Z")
    except (ValueError, OSError):
        return False
if pid_is_running(PID_FILE):
    print("이미 실행 중 PID:",PID_FILE.read_text().strip())
else:
    log=LOG_FILE.open("a",encoding="utf-8")
    process=subprocess.Popen(command,cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    PID_FILE.write_text(str(process.pid),encoding="utf-8")
    print("시작 PID:",process.pid,"로그:",LOG_FILE)

## 4. 실시간 진행률

In [ ]:
def read_json(path):
    try:return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError,json.JSONDecodeError):return {}
def fmt(seconds):
    if seconds is None:return "-"
    seconds=int(seconds);h,r=divmod(seconds,3600);m,s=divmod(r,60);return f"{h:02d}:{m:02d}:{s:02d}"
def pid_is_running(pid_file):
    if not pid_file.is_file():
        return False
    try:
        pid = int(pid_file.read_text().strip())
        result = subprocess.run(
            ["ps", "-o", "stat=", "-p", str(pid)],
            capture_output=True, text=True,
        )
        state = result.stdout.strip()
        return result.returncode == 0 and bool(state) and not state.startswith("Z")
    except (ValueError, OSError):
        return False
while True:
    from IPython.display import clear_output
    clear_output(wait=True);batch=read_json(BATCH_STATUS);rows=[]
    for name,path in PROGRESS_FILES.items():
        x=read_json(path);item=batch.get("folds",{}).get(name.split("_")[-1],{})
        rows.append({"Fold":name,"상태":x.get("status",item.get("status","waiting")),"Phase":item.get("phase","-"),"Epoch":f"{x.get('current_epoch',0)}/{x.get('total_epochs','-')}","진행률(%)":x.get("progress_percent",0),"ETA":fmt(x.get("eta_seconds")),"Dice":x.get("pseudo_dice"),"Recall":x.get("pseudo_recall"),"Best Composite":x.get("best_composite"),"Full Macro Dice":x.get("full_macro_dice"),"Best Full Macro Dice":x.get("best_full_macro_dice"),"Best Epoch":x.get("best_full_macro_epoch"),"Top-3":x.get("top_full_macro_epochs")})
    print("전체:",batch.get("status","starting"),"현재:",batch.get("current","-"),"PID:",pid_is_running(PID_FILE));display(pd.DataFrame(rows))
    if LOG_FILE.is_file():print("로그 마지막 10줄:
","".join(LOG_FILE.read_text(errors="replace").splitlines(True)[-10:]))
    if batch.get("status") in {"completed","failed"}:break
    time.sleep(10)

## 5. 완료 후 fold별 best 결과

In [ ]:
rows=[]
for fold in range(5):
    folder=BASE/f"fold_{fold}";best=read_json(folder/"best_full_macro.json")
    rows.append({"fold":fold,"best_epoch":best.get("epoch"),"macro_dice_positive":best.get("macro_dice_positive"),"checkpoint":best.get("checkpoint")})
results_df=pd.DataFrame(rows);display(results_df)
if results_df["macro_dice_positive"].notna().all():
    print("5-fold mean:",results_df["macro_dice_positive"].mean())
    print("5-fold std :",results_df["macro_dice_positive"].std(ddof=1))